- GEPA (Genetic-Pareto) 是一个通过“反思性文本进化” (Reflective Text Evolution) 来优化任意系统的框架。
    - 通过变异（Mutation）、反思（Reflection）和基于帕累托前沿的候选选择（Pareto-aware Candidate Selection）的迭代循环，GEPA 能够以较少的评估次数高效地进化出性能更优的文本组件。

- $C_i$: 代表第 $i$ 个候选者（candidate），它是一个从组件名称到文本内容的映射，例如 $C_i = \{\text{"prompt"}: \text{"..."}\}$。$\mathcal{P}$ 代表所有候选者的群体集合。
- $\mathcal{D}_{\text{train}}, \mathcal{D}_{\text{val}}$: 分别是训练数据集和验证数据集。
- $\mathcal{A}$: 代表 GEPAAdapter，它是一个接口，负责评估候选者并从执行中提取用于“反思”的轨迹。
    - A.evaluate
- $\mathcal{M}_{\text{reflect}}$: 用于反思和变异的强大语言模型（例如 GPT-4 或更高级的模型）。
- $B_{\text{max}}$: 最大的评估调用预算（budget）。
- $E$: 当前已使用的评估调用次数。
- $S_i$: 候选者 $C_i$ 在完整验证集 $\mathcal{D}_{\text{val}}$ 上的得分向量，其中 $S_{i,t}$ 是在第 $t$ 个验证样本上的得分。
- $\text{score}[i]$: 候选者 $C_i$ 在 $\mathcal{D}_{\text{val}}$ 上的聚合总分。
- $\mathcal{F}_t$: 对于验证集中的第 $t$ 个任务实例，处于帕累托前沿（即当前表现最好）的候选者索引集合。
- $f_t$: 到目前为止，在第 $t$ 个任务实例上取得的最高分。

### pareto fronts

- GEPA 在具体实现中，并不是计算一个全局的帕累托前沿，而是为验证集中的每一个任务实例（task instance）维护一个“最优候选集合”。这种实现方式更直接，也更高效。
    - GEPA引入了**帕累托前沿（Pareto Front）**的概念：它不只关注在所有任务上平均分最高的“全能冠军”，而是同时保留那些在某些特定子任务上表现出众的“单科冠军”。
- definitions
    - $\mathcal{P} = \{C_0, C_1, \dots, C_k\}$ 是当前所有候选者的集合。
    - $\mathcal{D}_{\text{val}} = \{d_1, d_2, \dots, d_N\}$ 是包含 $N$ 个任务实例的验证集。
    - $S_{i,t}$ 是候选者 $C_i$ 在任务实例 $d_t$ 上的得分。
- 对于每一个任务实例 $d_t$，我们首先计算出当前所有候选者在该任务上能达到的最高分 $f_t$:
    - $f_t=\max_{i\in\{0,\cdots,k\}}S_{i,t}$
- 然后，我们定义任务 $d_t$ 上的帕累托前沿集合（Pareto Front Set） $\mathcal{F}t$，它是所有在任务 $d_t$ 上取得了最高分 $f_t$ 的候选者的索引集合：
    - $F_t=\{i|S_{i,t}=f_t\}$